In [1]:
"""
fix_null_fields.py
==================
Tìm tất cả dish_ingredient có quantity_g, is_main, hoặc is_optional là NULL
→ Build prompt (dish_title + ingredient) → chia batch/file
→ AI điền 3 fields → import update bảng dish_ingredient
"""

import sqlite3
import json
import math
from pathlib import Path

DB_PATH    = "./cookpad_clean.db"
PROMPT_DIR = "./prompts_fix_null"
FILLED_DIR = "./filled_fix_null"
BATCH_SIZE = 30

PROMPT_TEMPLATE = """\
# ROLE
Bạn là chuyên gia ẩm thực Việt Nam với kinh nghiệm định lượng nguyên liệu chính xác.

# BỐI CẢNH
Các nguyên liệu dưới đây đang thiếu thông tin: quantity_g, is_main, is_optional.
Hãy điền các giá trị phù hợp cho **1 công thức nấu ăn gia đình (2–4 người ăn)**.

# QUY TẮC ĐỊNH LƯỢNG (quantity_g - đơn vị GRAM)
- Nguyên liệu chính (Thịt, Cá, Hải sản, Rau chính): 200g – 800g.
- Rau ăn kèm, củ quả phụ: 50g – 200g.
- Gia vị củ/quả (Hành, tỏi, ớt, gừng): 5g – 40g.
- Gia vị lá (Ngò, lá lốt, thì là): 5g – 20g.
- Nước dùng / Nước lọc: 500g – 2000g.
- Trả về số nguyên (int), KHÔNG > 2000 trừ nước dùng.

# QUY TẮC is_main (0 hoặc 1)
- 1: Nguyên liệu chính tạo nên món ăn (thịt, cá, rau chủ đạo, tinh bột chính).
- 0: Gia vị, nguyên liệu phụ, nước chấm, trang trí.

# QUY TẮC is_optional (0 hoặc 1)
- 1: Có thể bỏ qua mà món vẫn hoàn chỉnh (trang trí, gia vị tùy khẩu vị).
- 0: Bắt buộc phải có để món đúng công thức.

# LƯU Ý
- Nguyên liệu is_main=1 thường is_optional=0.
- Gia vị cơ bản (muối, đường, nước mắm) thường is_main=0, is_optional=0.
- Giữ tính logic giữa các nguyên liệu trong cùng một món.

# OUTPUT FORMAT
Chỉ trả về JSON array, KHÔNG kèm văn bản thừa:
[
  {{
    "id": <int>,
    "dish_title": "<string>",
    "ingredient_name": "<string>",
    "quantity_g": <int>,
    "is_main": <0 hoặc 1>,
    "is_optional": <0 hoặc 1>
  }}
]

## BATCH {batch_num}/{total_batches}
{payload_json}
"""

# ══════════════════════════════════════════════════════════════
# BƯỚC 1 — BUILD PROMPTS
# ══════════════════════════════════════════════════════════════

con = sqlite3.connect(DB_PATH)

rows = con.execute("""
    SELECT
        di.id,
        d.title       AS dish_title,
        i.name        AS ingredient_name,
        di.quantity_g,
        di.is_main,
        di.is_optional
    FROM dish_ingredient di
    JOIN dishes      d ON d.id = di.recipe_id
    JOIN ingredients i ON i.id = di.ingredient_id
    WHERE di.quantity_g IS NULL
       OR di.is_main    IS NULL
       OR di.is_optional IS NULL
    ORDER BY d.title, i.name
""").fetchall()

con.close()

print(f"Records có ít nhất 1 field NULL: {len(rows)}")

payload_all = [
    {
        "id":              r[0],
        "dish_title":      r[1],
        "ingredient_name": r[2],
        "quantity_g":      r[3],   # có thể null → AI sẽ điền
        "is_main":         r[4],   # có thể null → AI sẽ điền
        "is_optional":     r[5],   # có thể null → AI sẽ điền
    }
    for r in rows
]

total_items   = len(payload_all)
total_batches = math.ceil(total_items / BATCH_SIZE)

print(f"Batch size             : {BATCH_SIZE}")
print(f"Số file prompt sẽ tạo : {total_batches}")

Path(PROMPT_DIR).mkdir(exist_ok=True)
Path(FILLED_DIR).mkdir(exist_ok=True)

for i in range(total_batches):
    start = i * BATCH_SIZE
    end   = min(start + BATCH_SIZE, total_items)
    batch = payload_all[start:end]

    prompt = PROMPT_TEMPLATE.format(
        batch_num     = i + 1,
        total_batches = total_batches,
        payload_json  = json.dumps(batch, ensure_ascii=False, indent=2),
    )

    out_path = Path(PROMPT_DIR) / f"prompt_fix_null_batch_{i+1:02d}.txt"
    out_path.write_text(prompt, encoding="utf-8")
    print(f"  ✅ {out_path.name}  ({len(batch)} records)")

    # Tạo sẵn file JSON template rỗng để paste kết quả AI vào
    filled_path = Path(FILLED_DIR) / f"filled_fix_null_batch_{i+1:02d}.json"
    if not filled_path.exists():
        filled_path.write_text("[]", encoding="utf-8")

print(f"\n📁 Prompts : {Path(PROMPT_DIR).resolve()}")
print(f"📁 Filled  : {Path(FILLED_DIR).resolve()}")
print()
print("Next steps:")
print(f"  1. Mở từng file trong '{PROMPT_DIR}/'")
print(f"  2. Paste vào Claude / GPT-4o")
print(f"  3. Copy kết quả JSON vào file tương ứng trong '{FILLED_DIR}/'")
print(f"  4. Chạy BƯỚC 2 bên dưới")


# ══════════════════════════════════════════════════════════════
# BƯỚC 2 — IMPORT JSON → UPDATE dish_ingredient
# Chạy sau khi có đủ file filled_fix_null_batch_XX.json
# ══════════════════════════════════════════════════════════════

filled_files = sorted(Path(FILLED_DIR).glob("filled_fix_null_batch_*.json"))
filled_files = [f for f in filled_files if f.stat().st_size > 2]  # bỏ file rỗng "[]"

if not filled_files:
    print(f"❌ Chưa có file nào được điền trong '{FILLED_DIR}/'")
else:
    print(f"\nTìm thấy {len(filled_files)} file đã điền:")
    all_data = []
    for fpath in filled_files:
        with open(fpath, encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, dict):
            data = data.get("results") or list(data.values())[0]
        all_data.extend(data)
        print(f"  ✅ {fpath.name}: {len(data)} items")

    print(f"\nTổng items: {len(all_data)}")

    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()
    updated = 0
    skipped = 0

    for item in all_data:
        record_id   = item.get("id")
        quantity_g  = item.get("quantity_g")
        is_main     = item.get("is_main")
        is_optional = item.get("is_optional")

        if not record_id:
            skipped += 1
            continue

        # Chỉ update field nào AI đã điền (không ghi đè field đã có)
        fields, values = [], []

        if quantity_g is not None:
            try:
                q = int(quantity_g)
                if q > 0:
                    fields.append("quantity_g = ?")
                    values.append(q)
            except (ValueError, TypeError):
                pass

        if is_main is not None:
            fields.append("is_main = ?")
            values.append(int(is_main))

        if is_optional is not None:
            fields.append("is_optional = ?")
            values.append(int(is_optional))

        if not fields:
            skipped += 1
            continue

        values.append(record_id)
        cur.execute(
            f"UPDATE dish_ingredient SET {', '.join(fields)} WHERE id = ?",
            values
        )
        updated += cur.rowcount

    con.commit()
    con.close()

    print(f"\n✅ Updated : {updated}")
    print(f"⚠️  Skipped : {skipped}")

    # Verify
    con = sqlite3.connect(DB_PATH)
    row = con.execute("""
        SELECT
            SUM(CASE WHEN quantity_g  IS NULL THEN 1 ELSE 0 END) as null_qty,
            SUM(CASE WHEN is_main     IS NULL THEN 1 ELSE 0 END) as null_main,
            SUM(CASE WHEN is_optional IS NULL THEN 1 ELSE 0 END) as null_opt
        FROM dish_ingredient
    """).fetchone()
    con.close()

    print(f"\n=== Verify sau khi update ===")
    print(f"  Còn quantity_g  NULL : {row[0]}")
    print(f"  Còn is_main     NULL : {row[1]}")
    print(f"  Còn is_optional NULL : {row[2]}")

Records có ít nhất 1 field NULL: 10930
Batch size             : 30
Số file prompt sẽ tạo : 365
  ✅ prompt_fix_null_batch_01.txt  (30 records)
  ✅ prompt_fix_null_batch_02.txt  (30 records)
  ✅ prompt_fix_null_batch_03.txt  (30 records)
  ✅ prompt_fix_null_batch_04.txt  (30 records)
  ✅ prompt_fix_null_batch_05.txt  (30 records)
  ✅ prompt_fix_null_batch_06.txt  (30 records)
  ✅ prompt_fix_null_batch_07.txt  (30 records)
  ✅ prompt_fix_null_batch_08.txt  (30 records)
  ✅ prompt_fix_null_batch_09.txt  (30 records)
  ✅ prompt_fix_null_batch_10.txt  (30 records)
  ✅ prompt_fix_null_batch_11.txt  (30 records)
  ✅ prompt_fix_null_batch_12.txt  (30 records)
  ✅ prompt_fix_null_batch_13.txt  (30 records)
  ✅ prompt_fix_null_batch_14.txt  (30 records)
  ✅ prompt_fix_null_batch_15.txt  (30 records)
  ✅ prompt_fix_null_batch_16.txt  (30 records)
  ✅ prompt_fix_null_batch_17.txt  (30 records)
  ✅ prompt_fix_null_batch_18.txt  (30 records)
  ✅ prompt_fix_null_batch_19.txt  (30 records)
  ✅ prompt_f

In [3]:
import sqlite3

DB_PATH = "./cookpad_clean.db"

con = sqlite3.connect(DB_PATH)

rows = con.execute("""
    SELECT
        d.id,
        d.title,
        COUNT(di.ingredient_id) AS ingredient_count
    FROM dishes d
    JOIN dish_ingredient di ON di.recipe_id = d.id
    GROUP BY d.id
    HAVING COUNT(di.ingredient_id) < 3
    ORDER BY ingredient_count, d.title
""").fetchall()

print(f"Tìm thấy {len(rows)} món có < 3 nguyên liệu:\n")
print("=" * 60)

for dish_id, dish_title, ing_count in rows:
    ingredients = con.execute("""
        SELECT i.name, di.quantity_g, di.is_main
        FROM dish_ingredient di
        JOIN ingredients i ON i.id = di.ingredient_id
        WHERE di.recipe_id = ?
    """, (dish_id,)).fetchall()

    print(f"🍽️  {dish_title} (id={dish_id}) — {ing_count} nguyên liệu")
    for name, qty, is_main in ingredients:
        main_tag = "★" if is_main == 1 else " "
        qty_str  = f"{int(qty)}g" if qty else "NULL"
        print(f"    {main_tag} {name} — {qty_str}")
    print()

con.close()



Tìm thấy 1760 món có < 3 nguyên liệu:

🍽️  Bao tử heo luộc (id=16191994) — 1 nguyên liệu
    ★ bao tử heo — 200g

🍽️  Bánh papparoti nhân phô mai con bò cười (id=15566886) — 1 nguyên liệu
    ★ phô mai — 240g

🍽️  Buổi sáng thanh đạm:HÁ CẢO TÔM-RONG BIỂN (id=23846395) — 1 nguyên liệu
    ★ nhân bánh tôm — 200g

🍽️  Bánh Tráng Trộn Muối Tôm Đơn Giản (id=16155578) — 1 nguyên liệu
    ★ bánh tráng — 350g

🍽️  Bánh bò hấp nước cốt dừa (id=15531181) — 1 nguyên liệu
    ★ bột gạo — 130g

🍽️  Bánh bò làm từ men cơm rượu và đường mía (id=24258033) — 1 nguyên liệu
    ★ bột gạo — 100g

🍽️  Bánh bò rễ tre đường thốt nốt (id=16878323) — 1 nguyên liệu
    ★ bột gạo — 200g

🍽️  Bánh bò xốp (id=16872685) — 1 nguyên liệu
    ★ bột gạo — 200g

🍽️  Bánh bò ổ (id=24849973) — 1 nguyên liệu
    ★ bột gạo — 200g

🍽️  Bánh canh bột mì nấu thịt gà (id=16408965) — 1 nguyên liệu
    ★ gà — 600g

🍽️  Bánh canh cá nục nướng Đà Nẵng (id=16277955) — 1 nguyên liệu
    ★ cá nục — 1000g

🍽️  Bánh canh tôm mực (id=